# 🧠 CerebroSpheroid V5 Small: Cloud TPU High-Throughput Engine
### 105.9M Architecture • 100% Loss-Bearing Contiguous Pretraining • 15.7B Token Campaign

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_TPU_JAX.ipynb)

- **Pure JAX on Cloud TPU**: Native bfloat16 systolic MXU acceleration on TPU v5e / v6e.
- **In-Process Kernel Execution**: Invokes `run_training(...)` in-process, preventing TPU device lock aborts.
- **100% Loss-Bearing Contiguous Tokens**: Zero prompt-masking waste; unbroken 4,096-token sequence blocks.
- **Stage 1 Pretraining Mixture**: 85% FineWeb (sample-10BT) broad web + 15% Code (The Stack v2).
- **Cloud Sync Hub**: Checkpoints (`latest.pt`, `best.pt`) and live metrics stream to `bioflo.in` (no Google Drive required).

---
### ⚡ Colab Hardware Setting:
**Runtime** -> **Change runtime type** -> **TPU v5e** (TPU v5e-1) or **TPU v6e** (Trillium)

## Step 1: Environment Setup, HF Hub Auth & Repository Sync
Enables hugepages, detects Cloud TPU, logs into Hugging Face Hub (eliminating 429 rate limits), and synchronizes the latest codebase.

In [ ]:
# ==============================================================================
# 🚀 Step 1: Dependencies, Hugging Face Auth & Repository Sync
# ==============================================================================
!sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled" 2>/dev/null || true
!pip install -q transformers datasets sentencepiece optax torch pyarrow huggingface_hub

import os, sys, subprocess, jax

# 1. ⚡ Verify Cloud TPU Environment
devices = jax.devices()
dev_kind = devices[0].device_kind if devices else "CPU"
is_tpu = any("TPU" in d.device_kind.upper() for d in devices)
print(f"⚡ JAX Device: {dev_kind} ({len(devices)} device(s) active, Backend: {jax.default_backend().upper()})")
if is_tpu:
    print(f"✅ Cloud TPU Acceleration Active! ({dev_kind})")
else:
    print("⚠️ Running on CPU/GPU. For peak throughput, select Runtime -> Change runtime type -> TPU v5e.")

# 2. 🔑 Authenticate Hugging Face Hub (Eliminates 429 Too Many Requests)
try:
    from google.colab import userdata
    hf_tok = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN', '')
except Exception:
    hf_tok = os.environ.get('HF_TOKEN', '')

if hf_tok:
    os.environ['HF_TOKEN'] = hf_tok
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_tok
    try:
        from huggingface_hub import login
        login(token=hf_tok, add_to_git_credential=False)
        print("🔑 Hugging Face Hub authenticated successfully!")
    except Exception as e:
        print(f"ℹ️ Hugging Face login notice: {e}")

# 3. 🔄 Synchronize CerebroSpheroid V5 Codebase
REPO_DIR = '/content/cerebro_v5'
if not os.path.exists(REPO_DIR):
    print("📥 Cloning cerebro_spheroid_v5 repository...")
    subprocess.run(['git', 'clone', 'https://github.com/AjitKumar-git/cerebro_spheroid_v5.git', REPO_DIR], check=True)
else:
    print("🔄 Pulling latest updates from origin/main...")
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

rev = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
print(f"✅ Repository synchronized at commit [{rev}] in {REPO_DIR}!")

# 4. 📁 Ensure Checkpoint Directory Exists
CKPT_DIR = "/content/checkpoints/v5_small_scratch"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"📁 Local Checkpoint Directory Ready: {CKPT_DIR}")


## Step 2: Configure & Launch Training (In-Process)
Launches pretraining for `v5_small` (105.9M parameters) directly in-process via `run_training(...)`.
- **In-Process Execution**: Avoids TPU device locking collisions between parent Jupyter kernel and subprocesses.
- **100% Loss-Bearing Contiguous Tokens**: Continuous next-token prediction across full 4,096-token sequence blocks.
- **85% Web / 15% Code Mix**: Clean foundational pretraining on FineWeb 10BT and The Stack v2.
- **Full Dataset Ingestion**: 60,000 steps (~15.7 Billion tokens) with automatic bioflo.in checkpoint mirroring.

In [ ]:
# ==============================================================================
# ⚡ Step 2: Launch Contiguous Pretraining (In-Process)
# ==============================================================================
from scripts.train_v5_small_colab_tpu import run_training

# ── Primary Configuration ─────────────────────────────────────────────────────
run_training(
    mode="contiguous_pretrain",
    stage="pretrain",                  # 85% FineWeb 10BT + 15% Code (The Stack v2)
    total_window=4096,                 # Full 4,096-token context window
    sensory_position_steps=4096,
    max_phase_steps=4096,
    batch_size=8,                      # Micro-batch per core
    grad_accum=2,                      # Effective batch = 16 (65,536 tok/step)
    lr=2e-3,
    dtype="bf16",
    total_steps=60000,                 # 10x Full Ingestion: ~15.7 Billion tokens
    save_every=250,
    milestone_every=1000,
    probe_every=500,
    log_every=10,
    cloud_sync=True,
    resume=True,
    checkpoint_dir="/content/checkpoints/v5_small_scratch",
)


## Step 3: Monitor Checkpoints & Cloud Synchronization
Inspect local checkpoints and live sync status on the bioflo.in remote hub.

In [ ]:
# ==============================================================================
# 📊 Step 3: Checkpoint Inspector & Cloud Hub Dashboard
# ==============================================================================
import json
from src.cloud_sync import check_remote_manifest

print("📁 Local Saved Checkpoints:")
!ls -lh /content/checkpoints/v5_small_scratch

print("
🌐 Remote bioflo.in Cloud Hub Status (v5_small):")
try:
    manifest = check_remote_manifest(model="v5_small")
    print(json.dumps(manifest, indent=2))
except Exception as e:
    print(f"ℹ️ Cloud sync status notice: {e}")

print("
📈 Live Web Dashboard: https://bioflo.in/cerebro")


## Step 4: Interactive Live Inference & Prompt Testing
Generate completions from your trained checkpoint (locally cached or auto-pulled from bioflo.in).

In [ ]:
# ==============================================================================
# 💬 Step 4: Interactive Live Inference & Prompt Testing
# ==============================================================================
import os, jax
import jax.numpy as jnp
import numpy as np
from src.jax.v5_model import ModelConfigV5, PRESETS_V5, training_rollout
from src.jax.checkpoint import load_torch_pt_to_jax
from src.cloud_sync import pull_checkpoint
from src.corpus import build_tokenizer, format_stimulus_prompt, clean_motor_response, encode_no_special

CKPT_PATH = "/content/checkpoints/v5_small_scratch/v5_small_scratch_best.pt"
if not os.path.exists(CKPT_PATH):
    CKPT_PATH = "/content/checkpoints/v5_small_scratch/v5_small_scratch_latest.pt"
if not os.path.exists(CKPT_PATH):
    print("🌐 Checkpoint not found locally. Pulling best.pt from bioflo.in...")
    pull_checkpoint(CKPT_PATH, target_type="best", model="v5_small")

print(f"Loading checkpoint: {CKPT_PATH}...")
tokenizer = build_tokenizer("llama")

cfg_dict = PRESETS_V5["v5_small"].copy()
cfg_dict["sensory_position_steps"] = 4096
cfg_dict["max_phase_steps"] = 4096
cfg_dict["max_seq_len"] = 4096
config = ModelConfigV5(**cfg_dict)

params, buffers, meta = load_torch_pt_to_jax(CKPT_PATH, config, target_dtype=jnp.bfloat16)
print(f"✓ Loaded model from Step {meta.get('step', 0):,} (Loss: {meta.get('loss', 0.0):.4f})")

def generate(prompt_text: str, max_tokens: int = 128) -> str:
    stimulus = format_stimulus_prompt(prompt_text)
    tokens = encode_no_special(tokenizer, stimulus)
    prompt_arr = jnp.array([tokens], dtype=jnp.int32)
    
    out_logits = training_rollout(
        params=params,
        prompts=prompt_arr,
        target_len=max_tokens,
        buffers=buffers,
        config=config,
        targets=None,
        feedback_tokens=None,
        pure_teacher_forcing=False,
        deliberation_mode="teacher_forcing",
        repetition_penalty=1.15,
    )
    gen_token_ids = np.array(jnp.argmax(out_logits, axis=-1)[0])
    raw = tokenizer.decode(gen_token_ids, skip_special_tokens=False)
    return clean_motor_response(raw).strip()

test_prompt = "What is the capital of France?"
print(f"
Prompt:   {test_prompt}")
print(f"Response: {generate(test_prompt)}")
